<a href="https://colab.research.google.com/github/marcelonicolays/tcc_compras_diretas/blob/main/ML_TCC_compras_diretas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCP - Universidade Católica de Petrópolis
## Projeto de Conclusão de Curso
Marcelo Nicolay Santos

12510824

Dados extraídos no notebook ETL

In [6]:
import os
import numpy as np
import pandas as pd
import re
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
import warnings


In [2]:
df = pd.read_csv(
            "df_limpo.csv",
            sep=",",
            encoding="utf-8"
        )
print("Arquivo carregado com sucesso.")
print(df.head())

Arquivo carregado com sucesso.
                   Processo  ValorProcesso  \
0  **SEI-120001/009348/2022        10994.0   
1        020002/000001/2024          936.0   
2        020002/000002/2024         3696.0   
3        020002/000003/2024         5040.0   
4        020002/000004/2024         3120.0   

                                              Objeto  \
0  CONTRATAÇÃO DE PESSOA JURÍDICA ESPECIALIADA EM...   
1  PRESTAÇÃO DE SERVIÇOS DE CARÁTER CONTINUADO CO...   
2  GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...   
3  GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...   
4  GERAMOS O PROCESSO ADMINISTRATIVO PARA O EMPEN...   

                     Afastamento  AnoProcesso DataAprovacao  \
0    Dispensa - Pequenas Compras         2023    2023-07-05   
1  Inexigibilidade - 14.133/2021         2024    2024-02-08   
2  Inexigibilidade - 14.133/2021         2024    2024-02-08   
3  Inexigibilidade - 14.133/2021         2024    2024-02-08   
4  Inexigibilidade - 14.133/2021        

* Verificar Categorias

In [4]:
# Copia de segurança

df_limpo = df.copy()

# Auditoria inicial dos campos categóricos e textuais

colunas_auditoria = [
    'EnquadramentoLegal',
    'Unidade',
    'FornecedorVencedor',
    'Afastamento',
    'UnidadeMedida',
    'Objeto',
    'Item'
]

for coluna in colunas_auditoria:
    print(f"\n{'=' * 70}")
    print(f"COLUNA: {coluna}")
    print(f"Categorias distintas: {df_limpo[coluna].nunique(dropna=False)}")
    print("\n20 valores mais frequentes:")

    display(
        df_limpo[coluna]
        .value_counts(dropna=False)
        .head(20)
        .rename_axis('Valor')
        .reset_index(name='Frequência')
    )



COLUNA: EnquadramentoLegal
Categorias distintas: 1679

20 valores mais frequentes:


,Valor,Frequência
0,"Lei nº 14.133/2021, Art. 75º, II",2044
1,"Lei n 14.133/2021, Art. 74, I",1821
2,"Lei n 14.133/2021, Art. 75, II",1376
3,"Lei n 14.133/2021, Art. 74, II",1333
4,"Lei nº 14.133/2021, Art. 75º, VIII",1198
5,"Lei n 14.133/2021, Art. 74, III, f",914
6,"Lei n 14.133/2021, Art. 74, Caput",614
7,"Lei nº 14.133/2021, Art. 74º, I",486
8,"Em conformidade com o Inciso IV, do Art. 24 da...",442
9,"Lei 13.303/2016, Art. 29, II",405



COLUNA: Unidade
Categorias distintas: 99

20 valores mais frequentes:


,Valor,Frequência
0,UERJ - ADMINISTRACAO CENTRAL,2276
1,FUNARJ - FUND ANITA MANTUANO DE ARTES DO EST D...,1930
2,FSERJ - FUNDAÇÃO SAÚDE DO EST. DO RIO DE JANEIRO,1780
3,SEDSODH - SEC ESTADO DESEN SOCIAL E DE DIR HUM,1222
4,UERJ - HOSPITAL UNIVERSITÁRIO - HOSPITAL UNIVE...,1052
5,FMIS - FUNDAÇÃO MUSEU DA IMAGEM E DO SOM,638
6,NOVO DEGASE - Dep Geral de Ações Socioeducativas,624
7,SETUR - SECRETARIA DE TURISMO,599
8,SEEDUC - SECRETARIA DE ESTADO DE EDUCAÇÃO,536
9,SEPM - Secretaria de Estado de Polícia Militar,414



COLUNA: FornecedorVencedor
Categorias distintas: 4001

20 valores mais frequentes:


...

,Valor,Frequência
0,UN,9109
1,SERVICO,7424
2,VAGA,977
3,MENSAL,620
4,KG,419
5,MES,307
6,ANUAL,231
7,L,210
8,TESTE,174
9,PCT,154



COLUNA: Objeto
Categorias distintas: 10487

20 valores mais frequentes:


,Valor,Frequência
0,CONTRATAÇÃO EMERGENCIAL POR MENOR PREÇO GLOBAL...,150
1,CONTRATAÇÃO EMERGENCIAL POR MENOR PREÇO POR LO...,147
2,CONTRATAÇÃO EMERGENCIAL POR MENOR PREÇO GLOBAL...,123
3,AQUISIÇÃO DE PRODUTOS QUÍMICOS,119
4,CONTRATAÇÃO DE PRESTADOR DE SERVIÇOS PARA A GE...,111
5,CONTRATAÇÃO EMERGENCIAL POR MENOR PREÇO GLOBAL...,110
6,PARA CORROBORAR COM A MELHORIA DA AÇÃO DE APRE...,108
7,CONTRATAÇÃO EMERGENCIAL POR MENOR PREÇO GLOBAL...,106
8,CONTRATAÇÃO DE EMPRESA ESPECIALIZADA EM FORNEC...,85
9,AQUISIÇÃO DE MATERIAIS DE APOIO PEDAGÓGICO PAR...,84



COLUNA: Item
Categorias distintas: 7971

20 valores mais frequentes:


,Valor,Frequência
0,SERVICO DE PRODUCAO ARTISTICA - DESCRICAO: CON...,1800
1,SERVICOS DE ORGANIZACAO DE EVENTOS - DESCRIÇÃO...,485
2,SERVICO DE FORNECIMENTO DE AGUA - DESCRIÇÃO: C...,399
3,PARTICIPACAO EM CONGRESSOS / SIMPOSIO - DESCRI...,302
4,SERVICO DE FORNECIMENTO DE AGUA E COLETA DE ES...,276
5,SERVICOS DE APOIO ADMINISTRATIVO-FINANCEIRO - ...,229
6,SERVICO MEDICO-HOSPITALAR - DESCRIÇÃO: CONTRAT...,227
7,SERVICOS DE MANUTENCAO DE VEICULOS - DESCRIÇÃO...,200
8,SERVICO DE FORNECIMENTO DE AGUA E COLETA DE ES...,176
9,FORNECIMENTO DE ENERGIA ELETRICA - DESCRIÇÃO: ...,137


* Ajustando o enquadramento

In [7]:
def extrair_leis(texto):
    if pd.isna(texto) or not str(texto).strip():
        return None

    texto = str(texto).upper()
    encontradas = []

    # Padrões de leis conhecidas: número com ou sem pontuação/ano
    padroes = [
        (r'(?<!\d)14[\.\s]?133(?:\s*/\s*(?:2021|21))?(?!\d)', '14.133/2021'),
        (r'(?<!\d)13[\.\s]?303(?:\s*/\s*(?:2016|16))?(?!\d)', '13.303/2016'),
        (r'(?<!\d)8[\.,\s]?666(?:\s*/\s*(?:1993|93))?(?!\d)', '8.666/1993'),
        (r'(?<!\d)8666(?:\s*/\s*(?:1993|93))?(?!\d)', '8.666/1993'),
        (r'(?<!\d)10[\.\s]?520(?:\s*/\s*(?:2002|02))?(?!\d)', '10.520/2002'),
        (r'(?<!\d)9[\.\s]?970\s*/\s*2023(?!\d)', '9.970/2023'),
        (r'(?<!\d)10[\.\s]?277\s*(?:/\s*2024)?(?!\d)', '10.277/2024'),
        (r'(?<!\d)287\s*/\s*79(?!\d)', '287/1979'),
        (r'(?<!\d)5164\s*/\s*07(?!\d)', '5.164/2007'),
        (r'(?<!\d)6304\s*/\s*2012(?!\d)', '6.304/2012')
    ]

    for padrao, lei in padroes:
        if re.search(padrao, texto) and lei not in encontradas:
            encontradas.append(lei)

    if encontradas:
        return ' | '.join(encontradas)

    return None


# Gera a coluna final diretamente do texto original
df_limpo['LeiExtraida'] = df_limpo['EnquadramentoLegal'].apply(extrair_leis)

# Resumo da extração
print("Categorias distintas:", df_limpo['LeiExtraida'].nunique(dropna=True))
print("Registros sem lei identificada:", df_limpo['LeiExtraida'].isna().sum())

print("\nTop 20 — Leis identificadas:")
display(
    df_limpo['LeiExtraida']
    .value_counts()
    .head(20)
    .rename_axis('Lei')
    .reset_index(name='Quantidade')
)

print("\nTop 30 — Enquadramentos sem lei identificada:")
display(
    df_limpo.loc[df_limpo['LeiExtraida'].isna(), 'EnquadramentoLegal']
    .fillna('(Enquadramento vazio)')
    .value_counts()
    .head(30)
    .rename_axis('Enquadramento original')
    .reset_index(name='Quantidade')
)

Categorias distintas: 13
Registros sem lei identificada: 415

Top 20 — Leis identificadas:


,Lei,Quantidade
0,14.133/2021,12267
1,8.666/1993,6393
2,13.303/2016,1494
3,9.970/2023,14
4,287/1979,12
5,13.303/2016 | 8.666/1993,10
6,5.164/2007 | 6.304/2012,9
7,10.520/2002,9
8,10.277/2024,8
9,8.666/1993 | 10.520/2002,6



Top 30 — Enquadramentos sem lei identificada:


,Enquadramento original,Quantidade
0,Lei 13.133/16 art.29 inc.II,34
1,Pequenas compras,28
2,decreto 47.364/2020,20
3,pequenas compras,15
4,"Em conformidade com o Art.24, Inciso IV da Lei...",14
5,"Inexigibilidade de licitação, Art. 24, decreto...",14
6,DISPENSA DE LICITAÇÃO,13
7,CONTRATAÇÃO DE REMANESCENTE DE SERVIÇO. ART. 2...,12
8,CONTRATAÇÃO DE REMANESCENTE DE SERVIÇO. ART. 2...,12
9,lei 866 n aplicvavel,11


In [ ]:
"""
Pipeline de Machine Learning Não Supervisionado para Detecção de Anomalias
em Compras Diretas do Estado do Rio de Janeiro (contasRJ)

Metodologia:
1. Pré-processamento e Normalização (RobustScaler / StandardScaler)
2. Análise de Agrupamento (K-Means) com busca de K via Método do Cotovelo e Silhueta (K >= 20)
3. Avaliação de Divergência entre Clusters (tamanho do grupo, distância do centroide global, desvio de preços)
4. Detecção de Outliers Multidimensionais com Isolation Forest
5. Matriz de Risco e Consenso de Anomalias (Cluster Divergente + Isolation Forest)
"""

warnings.filterwarnings('ignore')

def preparar_features(df):
    """
    Realiza a engenharia de atributos e preparação dos dados numéricos e categóricos.
    """
    df_prep = df.copy()

    # Tratamento de valores nulos e conversão de tipos
    for col in ['ValorProcesso', 'ValorUnitario', 'QuantidadeItem', 'ValorTotalItem']:
        if col in df_prep.columns:
            df_prep[col] = pd.to_numeric(df_prep[col], errors='coerce').fillna(0)

    # Criar transformações logarítmicas para atenuar assimetria financeira
    df_prep['log_valor_unitario'] = np.log1p(np.maximum(0, df_prep.get('ValorUnitario', 0)))
    df_prep['log_valor_processo'] = np.log1p(np.maximum(0, df_prep.get('ValorProcesso', 0)))
    df_prep['log_quantidade'] = np.log1p(np.maximum(0, df_prep.get('QuantidadeItem', 0)))

    # Razão entre valor do item e valor total do processo (se disponíveis)
    if 'ValorTotalItem' in df_prep.columns and 'ValorProcesso' in df_prep.columns:
        df_prep['proporcao_item_processo'] = np.where(
            df_prep['ValorProcesso'] > 0,
            df_prep['ValorTotalItem'] / df_prep['ValorProcesso'],
            0
        )
    else:
        df_prep['proporcao_item_processo'] = 0.0

    features_numericas = ['log_valor_unitario', 'log_valor_processo', 'log_quantidade', 'proporcao_item_processo']

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(df_prep[features_numericas])

    return df_prep, X_scaled, features_numericas, scaler

def avaliar_cotovelo_silhueta(X_scaled, k_range=range(2, 31, 2), sample_size=5000):
    """
    Executa testes de K-Means para diferentes valores de K, calculando a Inércia (Cotovelo)
    e o Coeficiente de Silhueta.
    """
    print("--- Executando Análise de Cotovelo e Silhueta ---")
    resultados = []

    # Subamostragem para cálculo rápido da silhueta caso o dataset seja volumoso
    n_samples = min(sample_size, X_scaled.shape[0])
    indices_sub = np.random.choice(X_scaled.shape[0], size=n_samples, replace=False) if X_scaled.shape[0] > sample_size else np.arange(X_scaled.shape[0])
    X_sub = X_scaled[indices_sub]

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        inercia = kmeans.inertia_

        sil = silhouette_score(X_sub, labels[indices_sub])

        resultados.append({
            'k': k,
            'inercia': inercia,
            'silhueta': sil
        })
        print(f"K = {k:2d} | Inércia: {inercia:12.2f} | Silhueta: {sil:.4f}")

    return pd.DataFrame(resultados)

def treinar_clustering_divergente(df_prep, X_scaled, n_clusters=20):
    """
    Treina o modelo K-Means com K selecionado (mínimo recomendado de 20 clusters)
    e identifica os clusters mais divergentes/anômalos.
    """
    print(f"\n--- Treinando K-Means com K = {n_clusters} ---")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=15)
    labels = kmeans.fit_predict(X_scaled)
    centroids = kmeans.cluster_centers_

    # Calcular distância de cada ponto ao seu próprio centroide
    distancias_centroide_proprio = np.linalg.norm(X_scaled - centroids[labels], axis=1)

    # Calcular centroide global para medir a divergência do grupo
    centroide_global = np.mean(X_scaled, axis=0)
    distancia_clusters_global = np.linalg.norm(centroids - centroide_global, axis=1)

    # Analisar perfil de cada cluster
    df_prep['cluster_id'] = labels
    df_prep['dist_centroide_proprio'] = distancias_centroide_proprio

    cluster_metrics = []
    total_amostras = len(df_prep)

    for c_id in range(n_clusters):
        mask = (labels == c_id)
        tamanho = np.sum(mask)
        percentual = (tamanho / total_amostras) * 100

        # Média e mediana dos preços unitários originais do cluster
        val_unit_med = df_prep.loc[mask, 'ValorUnitario'].median() if 'ValorUnitario' in df_prep.columns else 0
        val_unit_mean = df_prep.loc[mask, 'ValorUnitario'].mean() if 'ValorUnitario' in df_prep.columns else 0

        dist_global = distancia_clusters_global[c_id]

        # Score de Divergência do Cluster (combina raridade do grupo e distância do centroide global)
        # Clusters pequenos e afastados do centro populacional recebem maior pontuação de divergência
        score_divergencia = dist_global * (1 / np.log2(tamanho + 2))

        cluster_metrics.append({
            'cluster_id': c_id,
            'tamanho_amostras': tamanho,
            'percentual_base': percentual,
            'distancia_centro_global': dist_global,
            'valor_unitario_medio': val_unit_mean,
            'valor_unitario_mediana': val_unit_med,
            'score_divergencia_cluster': score_divergencia
        })

    df_clusters = pd.DataFrame(cluster_metrics).sort_values(by='score_divergencia_cluster', ascending=False)

    # Classificar os N clusters de maior divergência (top 20%)
    top_divergentes = set(df_clusters.head(int(np.ceil(n_clusters * 0.2)))['cluster_id'])
    df_prep['flag_cluster_divergente'] = df_prep['cluster_id'].isin(top_divergentes).astype(int)

    return df_prep, kmeans, df_clusters

def treinar_isolation_forest(X_scaled, contamination=0.05):
    """
    Aplica o algoritmo Isolation Forest para detectar anomalias multidimensionais
    sem necessidade de rótulos prévios.
    """
    print(f"\n--- Treinando Isolation Forest (Contaminação esperada: {contamination*100:.1f}%) ---")
    iso_forest = IsolationForest(contamination=contamination, random_state=42, n_jobs=-1)

    # Isolation forest retorna -1 para anomalias e 1 para inliers
    preds = iso_forest.fit_predict(X_scaled)
    flags_anomalia = np.where(preds == -1, 1, 0)

    # Score de decisão inverter sinal (quanto mais negativo/alto, mais anômalo)
    decision_scores = -iso_forest.score_samples(X_scaled)

    return flags_anomalia, decision_scores, iso_forest

def consolidar_consenso_anomalias(df_prep, df_clusters):
    """
    Consolida os resultados dos modelos não supervisionados (Clustering + Isolation Forest)
    criando uma matriz de risco isenta de viés de rotulagem sintética interna.
    """
    print("\n--- Consolidando Matriz de Risco Não Supervisionada ---")

    # Normalização min-max do score do Isolation Forest para escala 0 - 100
    min_s, max_s = df_prep['iforest_score'].min(), df_prep['iforest_score'].max()
    score_if_norm = ((df_prep['iforest_score'] - min_s) / (max_s - min_s + 1e-9)) * 100

    # Mapear score de divergência do cluster para cada registro
    mapa_divergencia = dict(zip(df_clusters['cluster_id'], df_clusters['score_divergencia_cluster']))
    df_prep['score_divergencia_cluster'] = df_prep['cluster_id'].map(mapa_divergencia)

    min_d, max_d = df_prep['score_divergencia_cluster'].min(), df_prep['score_divergencia_cluster'].max()
    score_cluster_norm = ((df_prep['score_divergencia_cluster'] - min_d) / (max_d - min_d + 1e-9)) * 100

    # Score Consolidado de Suspeição (50% Divergência do Cluster + 50% Isolation Forest)
    df_prep['score_suspeicao_nao_supervisionado'] = (0.5 * score_cluster_norm) + (0.5 * score_if_norm)

    # Flag de Consenso de Anomalia (Quando pertence a cluster divergente E é marcado pelo Isolation Forest)
    df_prep['consenso_anomalia'] = np.where(
        (df_prep['flag_cluster_divergente'] == 1) & (df_prep['flag_iforest_anomalia'] == 1),
        1,
        0
    )

    print("Resumo do Consenso de Anomalias:")
    print(f"Total de Registros Analisados: {len(df_prep)}")
    print(f"Registros em Clusters Divergentes: {df_prep['flag_cluster_divergente'].sum()}")
    print(f"Anomalias apontadas pelo Isolation Forest: {df_prep['flag_iforest_anomalia'].sum()}")
    print(f"Anomalias com Consenso Duplo (Cluster + IF): {df_prep['consenso_anomalia'].sum()}")

    return df_prep

def executar_pipeline_completo(caminho_csv='df_limpo.csv', n_clusters_escolhido=20):
    """
    Função principal para carregar os dados, executar avaliação de K,
    clusterização com N >= 20, Isolation Forest e salvar o dataset enriquecido.
    """
    if not os.path.exists(caminho_csv):
        print(f"Arquivo {caminho_csv} não encontrado. Gerando dados sintéticos para demonstração técnica...")
        np.random.seed(42)
        n = 1000
        df_dummy = pd.DataFrame({
            'Processo': [f"PROC-{i//5}" for i in range(n)],
            'ValorProcesso': np.random.exponential(scale=50000, size=n),
            'ValorUnitario': np.random.exponential(scale=200, size=n),
            'QuantidadeItem': np.random.randint(1, 100, size=n),
            'ValorTotalItem': np.random.exponential(scale=1000, size=n)
        })
        # Injetar alguns outliers deliberados
        df_dummy.loc[10:15, 'ValorUnitario'] *= 50
        df_prep, X_scaled, features, scaler = preparar_features(df_dummy)
    else:
        df_raw = pd.read_csv(caminho_csv)
        df_prep, X_scaled, features, scaler = preparar_features(df_raw)

    # 1. Teste de Cotovelo e Silhueta para escolha do K
    df_avaliacoes_k = avaliar_cotovelo_silhueta(X_scaled, k_range=range(5, 31, 5))

    # 2. Clusterização com K >= 20
    df_prep, model_kmeans, df_clusters = treinar_clustering_divergente(df_prep, X_scaled, n_clusters=n_clusters_escolhido)

    # 3. Isolation Forest
    flags_if, scores_if, model_if = treinar_isolation_forest(X_scaled, contamination=0.05)
    df_prep['flag_iforest_anomalia'] = flags_if
    df_prep['iforest_score'] = scores_if

    # 4. Consolidar Matriz de Risco Não Supervisionada
    df_resultado = consolidar_consenso_anomalias(df_prep, df_clusters)

    # 5. Salvar Resultados
    caminho_saida = 'df_ml_nao_supervisionado_resultados.csv'
    df_resultado.to_csv(caminho_saida, index=False)
    print(f"\n✅ Processo concluído! Dataset enriquecido salvo em '{caminho_saida}'.")

    return df_resultado, df_clusters, df_avaliacoes_k

if __name__ == '__main__':
    executar_pipeline_completo()
